# ARCHIVED RUN RECORD: DO NOT RUN

This is the executed notebook from the **2026-11-11** Google Colab run of the Oxford Flowers-102 exploration, kept as evidence of the results (all 10 split checks passed).

**It is superseded** by `notebooks/01_dataset_exploration_sibol.ipynb`. This copy contains the original workaround cells, which:
- wrote a simplified copy of `src/data/flowers102.py` into `/content/sibol` (the GitHub repository was empty), and
- fell back to `download=True` when metadata was missing, which downloaded `imagelabels.mat` and `setid.mat` from the internet.

Both problems are fixed in the current notebook. Text outputs below are unchanged. Embedded figures were replaced by references to the identical PNG files in `outputs/figures/`.
Written up in `docs/dataset_exploration_report.md` and `Documentation.md`.

### Option A: Clone directly into the local Colab workspace (Recommended & Simplest)
This is fast and does not use your Google Drive space. However, files will be deleted when this Colab session ends.

### Option B: Clone permanently to your Google Drive
This clones the files into your Google Drive under a folder named `sibol` so your work is saved even after the Colab session is closed.

In [7]:
# Replace <YOUR_GITHUB_REPO_URL> with your actual SIBOL repository link.
# Example: https://github.com/your-username/sibol.git

REPO_URL = "https://github.com/didyouseejay/sibol.git"

# --- OPTION A: Clone to Local Colab Space (Recommended) ---
!git clone {REPO_URL} /content/sibol

# --- OPTION B: Clone to Google Drive (Permanent) ---
# Uncomment the two lines below if you want to save it directly to Google Drive instead:
# import os
# !git clone {REPO_URL} "/content/drive/MyDrive/sibol"

Cloning into '/content/sibol'...


In [8]:
import os

# Create the directory structure locally in /content/sibol
os.makedirs("/content/sibol/src/data", exist_ok=True)

# Write the __init__.py files
with open("/content/sibol/src/__init__.py", "w") as f:
    pass
with open("/content/sibol/src/data/__init__.py", "w") as f:
    pass

# Write the flowers102.py helper script with expected dataset methods
flowers_code = """
import os
from pathlib import Path
import torchvision

NUM_CLASSES = 102
SPLITS = ['train', 'val', 'test']
EXPECTED_TOTAL_IMAGES = 8189
EXPECTED_SPLIT_SIZES = {'train': 1020, 'val': 1020, 'test': 6149}
EXPECTED_IMAGES_PER_CLASS = {'train': 10, 'val': 10}

def default_data_root(in_colab, project_root):
    if in_colab:
        return Path("/content/data")
    return Path(project_root) / "data"

def is_downloaded(data_root):
    # Quick check if the dataset directory exists and has files
    flower_dir = Path(data_root) / "flowers-102"
    return flower_dir.exists() and len(list(flower_dir.glob("**/*"))) > 0

def load_split(data_root, split, download=False):
    # Map standard split names to torchvision equivalents
    tv_split = 'test' if split == 'test' else ('train' if split == 'train' else 'val')
    return torchvision.datasets.Flowers102(
        root=str(data_root),
        split=tv_split,
        download=download
    )

def get_labels(dataset):
    return dataset._labels

def get_image_files(dataset):
    return dataset._image_files

def class_counts(labels):
    counts = [0] * NUM_CLASSES
    for label in labels:
        counts[label] += 1
    return counts
"""

with open("/content/sibol/src/data/flowers102.py", "w") as f:
    f.write(flowers_code.strip())

print("Project structure and helper files successfully created inside '/content/sibol/src'!")


Project structure and helper files successfully created inside '/content/sibol/src'!


In [10]:
import shutil
from pathlib import Path

# Let's search dynamically for any .tgz files in the Colab workspace
search_paths = list(Path("/content").rglob("*.tgz"))

source_file = None
for path in search_paths:
    if "102flowers" in path.name or "flowers102" in path.name:
        source_file = path
        break

# Destination where torchvision expects it
dest_dir = Path("/content/data/flowers-102")
dest_dir.mkdir(parents=True, exist_ok=True)

if source_file:
    dest_file = dest_dir / "102flowers.tgz"
    if not dest_file.exists():
        shutil.copy(source_file, dest_file)
        print(f"Successfully found and copied dataset archive from {source_file} to {dest_file}!")
    else:
        print(f"Dataset archive already exists at {dest_file}")
else:
    print("Could not find any matching .tgz file inside /content. Please upload the file or ensure it is downloaded.")

Successfully found and copied dataset archive from /content/drive/MyDrive/sibol/src/102flowers.tgz to /content/data/flowers-102/102flowers.tgz!


# 01 · Oxford Flowers-102: Dataset Exploration

**SIBOL: System for Image-Based Observation and Learning**

This notebook is a **scaffold** for exploring the dataset that all three approaches (HOG + Linear SVM, custom CNN, ResNet-18) will share:

1. Load the **official** train / validation / test splits.
2. **Verify** the splits against the dataset authors' documented sizes.
3. Plot the **class distribution** of each split.
4. Show **sample images** with their class names.
5. Measure **image size statistics** (to guide resizing choices later).

**Nothing is downloaded by default.** If the dataset is not found, every analysis cell prints a message and skips itself, so the notebook still runs top to bottom. Set `DOWNLOAD = True` in the configuration cell when you are ready to fetch the data (about 330 MB).

Dataset: Nilsback & Zisserman, *Automated Flower Classification over a Large Number of Classes*, ICVGIP 2008. [Project page](https://www.robots.ox.ac.uk/~vgg/data/flowers/102/).

## 0. Setup: locate the project and import helpers

The reusable dataset code lives in `src/data/flowers102.py`. We add the project root to `sys.path` so the notebook can import it.

**In Colab**, first put the SIBOL repository on Google Drive (or clone it into `/content`) and set `COLAB_PROJECT_ROOT` below to its location.

In [11]:
import importlib.util
import random
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None

# Point explicitly to where we initialized your SIBOL project structure
COLAB_PROJECT_ROOT = Path("/content/sibol")

if IN_COLAB:
    # Use the local clone path as it has the generated helper codes under src/
    PROJECT_ROOT = COLAB_PROJECT_ROOT
else:
    # Walk upwards from the notebook's folder until we find the folder that contains src/.
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(
        f"Could not find the SIBOL 'src' folder under {PROJECT_ROOT}. "
        f"Please verify that the repository files exist in that path."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import flowers102 as f102

print(f"Running in Colab: {IN_COLAB}")
print(f"Project root    : {PROJECT_ROOT}")

Running in Colab: True
Project root    : /content/sibol


## 1. Configuration

In [12]:
# Where the dataset lives (Drive in Colab, <project>/data locally; git-ignored).
DATA_ROOT = f102.default_data_root(IN_COLAB, PROJECT_ROOT)

# Keep False until you are ready to download (~330 MB).
DOWNLOAD = False

# Where generated figures are saved (git-ignored).
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42  # makes the random sample images reproducible

print(f"Data root : {DATA_ROOT}")
print(f"Downloaded: {f102.is_downloaded(DATA_ROOT)}")
print(f"DOWNLOAD  : {DOWNLOAD}")

Data root : /content/data
Downloaded: True
DOWNLOAD  : False


## 2. Load the official splits

`torchvision.datasets.Flowers102` reads the official split file (`setid.mat`) published by the dataset authors, so we use exactly the splits used in the literature.

In [14]:
datasets = {}

# We check if the main archive is present
if f102.is_downloaded(DATA_ROOT) or DOWNLOAD or (DATA_ROOT / "flowers-102/102flowers.tgz").exists():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    for split in f102.SPLITS:
        try:
            # Try loading normally first
            datasets[split] = f102.load_split(DATA_ROOT, split, download=DOWNLOAD)
        except RuntimeError:
            # If .mat metadata files are missing, force download=True to fetch only the tiny metadata files (<1MB)
            print(f"Fetching missing metadata for {split} split...")
            datasets[split] = f102.load_split(DATA_ROOT, split, download=True)
        print(f"Loaded {split:<5}: {len(datasets[split])} images")
else:
    print("Dataset not found and DOWNLOAD = False, so nothing was loaded.")
    print("Set DOWNLOAD = True in the configuration cell when you are ready.")

DATA_READY = len(datasets) == len(f102.SPLITS)

# Class names: newer torchvision versions provide them; otherwise use numbered names.
CLASS_NAMES = None
if DATA_READY:
    CLASS_NAMES = getattr(datasets["train"], "classes", None)
if not CLASS_NAMES:
    CLASS_NAMES = [f"class_{i:03d}" for i in range(f102.NUM_CLASSES)]

Fetching missing metadata for train split...


100%|██████████| 502/502 [00:00<00:00, 1.66MB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 42.2MB/s]


Loaded train: 1020 images
Loaded val  : 1020 images
Loaded test : 6149 images


## 3. Verify the official splits

Each check compares what we loaded against the dataset authors' documented values:

| Check | Expected |
|---|---|
| Split sizes | train 1020, val 1020, test 6149 |
| Total images | 8189 |
| Number of classes in every split | 102 |
| Images per class in train and val | exactly 10 |
| Splits overlap? | no shared images |

In [15]:
def report(check_name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {check_name}" + (f": {detail}" if detail else ""))
    return passed


if not DATA_READY:
    print("Skipped: dataset not loaded.")
else:
    labels = {s: f102.get_labels(ds) for s, ds in datasets.items()}
    files = {s: set(f102.get_image_files(ds)) for s, ds in datasets.items()}
    results = []

    for s in f102.SPLITS:
        expected = f102.EXPECTED_SPLIT_SIZES[s]
        results.append(report(f"{s} size", len(datasets[s]) == expected,
                              f"{len(datasets[s])} (expected {expected})"))

    total = sum(len(ds) for ds in datasets.values())
    results.append(report("total images", total == f102.EXPECTED_TOTAL_IMAGES,
                          f"{total} (expected {f102.EXPECTED_TOTAL_IMAGES})"))

    for s in f102.SPLITS:
        n_classes = len(set(labels[s]))
        results.append(report(f"{s} classes", n_classes == f102.NUM_CLASSES,
                              f"{n_classes} (expected {f102.NUM_CLASSES})"))

    for s, per_class in f102.EXPECTED_IMAGES_PER_CLASS.items():
        counts = f102.class_counts(labels[s])
        results.append(report(f"{s} images per class", all(c == per_class for c in counts),
                              f"min {min(counts)}, max {max(counts)} (expected {per_class})"))

    if all(files.values()):
        overlap = (files["train"] & files["val"]) | (files["train"] & files["test"]) | (files["val"] & files["test"])
        results.append(report("splits are disjoint", not overlap, f"{len(overlap)} shared files"))
    else:
        print("[SKIP] disjointness check: image file names not exposed by this torchvision version.")

    print()
    print("ALL CHECKS PASSED" if all(results) else "SOME CHECKS FAILED: investigate before training.")

[PASS] train size: 1020 (expected 1020)
[PASS] val size: 1020 (expected 1020)
[PASS] test size: 6149 (expected 6149)
[PASS] total images: 8189 (expected 8189)
[PASS] train classes: 102 (expected 102)
[PASS] val classes: 102 (expected 102)
[PASS] test classes: 102 (expected 102)
[PASS] train images per class: min 10, max 10 (expected 10)
[PASS] val images per class: min 10, max 10 (expected 10)
[PASS] splits are disjoint: 0 shared files

ALL CHECKS PASSED


## 4. Class distribution

Train and val are balanced by design (10 per class). The test split holds all remaining images, so it is **imbalanced**. That matters when we report accuracy later (we should also look at per-class / macro-averaged metrics).

In [16]:
import matplotlib.pyplot as plt
import numpy as np

if not DATA_READY:
    print("Skipped: dataset not loaded.")
else:
    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
    for ax, s in zip(axes, f102.SPLITS):
        counts = np.array(f102.class_counts(labels[s]))
        ax.bar(np.arange(f102.NUM_CLASSES), counts, width=0.85)
        ax.set_title(f"{s}: {counts.sum()} images | min {counts.min()}, "
                     f"max {counts.max()}, mean {counts.mean():.1f} per class")
        ax.set_ylabel("images")
    axes[-1].set_xlabel("class index (0-101)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "class_distribution.png", dpi=120)
    plt.show()

    # Most and least frequent classes in the test split.
    test_counts = np.array(f102.class_counts(labels["test"]))
    order = np.argsort(test_counts)
    print("Fewest test images:", [(CLASS_NAMES[i], int(test_counts[i])) for i in order[:5]])
    print("Most test images  :", [(CLASS_NAMES[i], int(test_counts[i])) for i in order[-5:][::-1]])

[Figure removed from this archived copy to keep the repository small.
 Byte-identical PNG: outputs/figures/class_distribution.png]

Fewest test images: [('pink primrose', 20), ('canterbury bells', 20), ('moon orchid', 20), ('fire lily', 20), ('prince of wales feathers', 20)]
Most test images  : [('petunia', 238), ('passion flower', 231), ('wallflower', 176), ('water lily', 174), ('watercress', 164)]


## 5. Sample images

A random grid from the training split (fixed seed, so the grid is reproducible).

In [17]:
if not DATA_READY:
    print("Skipped: dataset not loaded.")
else:
    rng = random.Random(SEED)
    idx = rng.sample(range(len(datasets["train"])), 16)

    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    for ax, i in zip(axes.flat, idx):
        image, label = datasets["train"][i]  # PIL image (no transform applied)
        ax.imshow(image)
        ax.set_title(f"{label}: {CLASS_NAMES[label]}", fontsize=9)
        ax.axis("off")
    fig.suptitle("Random training samples", fontsize=14)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "sample_images.png", dpi=120)
    plt.show()

[Figure removed from this archived copy to keep the repository small.
 Byte-identical PNG: outputs/figures/sample_images.png]

### 5b. Several examples of one class

Useful for seeing intra-class variation (pose, scale, colour, background). Change `CLASS_ID` to look at other classes.

In [18]:
CLASS_ID = 0

if not DATA_READY:
    print("Skipped: dataset not loaded.")
else:
    members = [i for i, y in enumerate(labels["train"]) if y == CLASS_ID]
    fig, axes = plt.subplots(1, len(members), figsize=(2.2 * len(members), 2.6))
    for ax, i in zip(np.atleast_1d(axes), members):
        ax.imshow(datasets["train"][i][0])
        ax.axis("off")
    fig.suptitle(f"Class {CLASS_ID}: {CLASS_NAMES[CLASS_ID]} (all train images)")
    fig.tight_layout()
    plt.show()

[Figure removed from this archived copy to keep the repository small.
 Byte-identical PNG: outputs/figures/class_000_examples.png]

## 6. Image size statistics

Flowers-102 images have varying sizes. We check a random subset so we can choose sensible input sizes for HOG (fixed size needed) and the CNNs.

In [19]:
N_SIZE_SAMPLES = 300

if not DATA_READY:
    print("Skipped: dataset not loaded.")
else:
    rng = random.Random(SEED)
    split_ds = datasets["train"]
    pick = rng.sample(range(len(split_ds)), min(N_SIZE_SAMPLES, len(split_ds)))
    sizes = np.array([split_ds[i][0].size for i in pick])  # (width, height)
    widths, heights = sizes[:, 0], sizes[:, 1]
    aspect = widths / heights

    print(f"Sampled {len(pick)} training images")
    print(f"Width : min {widths.min()}, median {np.median(widths):.0f}, max {widths.max()}")
    print(f"Height: min {heights.min()}, median {np.median(heights):.0f}, max {heights.max()}")
    print(f"Aspect ratio (w/h): min {aspect.min():.2f}, median {np.median(aspect):.2f}, max {aspect.max():.2f}")

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(widths, heights, s=8, alpha=0.6)
    ax.set_xlabel("width (px)")
    ax.set_ylabel("height (px)")
    ax.set_title("Image sizes (random training subset)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "image_sizes.png", dpi=120)
    plt.show()

Sampled 300 training images
Width : min 500, median 667, max 857
Height: min 500, median 500, max 861
Aspect ratio (w/h): min 0.58, median 1.33, max 1.71


[Figure removed from this archived copy to keep the repository small.
 Byte-identical PNG: outputs/figures/image_sizes.png]

## 7. Observations

*Recorded on 2026-11-11 from the outputs of this run (Google Colab, data root `/content/data`). Everything below is taken from the outputs above; interpretations are marked as such.*

**Split verification**
- All 10 checks passed: train 1020, val 1020, test 6149 (total 8189); 102 classes in every split; exactly 10 images per class in train and val; 0 image files shared between splits.
- The official splits loaded by `torchvision.datasets.Flowers102` match the dataset authors' documentation, so they can be used as-is for all three approaches.

**Class balance**
- Train and val are perfectly balanced (10 images per class).
- Test is imbalanced: min 20, max 238, mean 60.3 images per class (about 12x between the smallest and largest class).
- Fewest test images (20 each): pink primrose, canterbury bells, moon orchid, fire lily, prince of wales feathers.
- Most test images: petunia (238), passion flower (231), wallflower (176), water lily (174), watercress (164).
- *Implication:* overall test accuracy is weighted towards the large classes, so we should also report macro-averaged and per-class metrics.

**Visual characteristics** (from the 16-image random grid and the 10 pink primrose training images)
- Backgrounds are mostly natural and cluttered (leaves, grass, soil, fences), not plain studio backgrounds.
- Some images contain several flowers, or flowers alongside buds (e.g. artichoke, pink primrose).
- Colour varies within a class: the three morning glory samples are pink, white/lilac and blue; pink primrose ranges from almost white to deep pink.
- Scale and framing vary: some flowers fill the frame (corn poppy, colt's foot), others are small within the scene (cautleya spicata, fritillary).
- Several images carry photographer watermarks or copyright text (e.g. corn poppy, yellow iris, tree poppy, one pink primrose image).

**Image sizes** (random sample of 300 training images, seed 42)
- Width 500 to 857 px (median 667); height 500 to 861 px (median 500); aspect ratio (w/h) 0.58 to 1.71 (median 1.33).
- The scatter plot shows every sampled image has its shorter side equal to 500 px: landscape images are 500 px tall, portrait images are 500 px wide.
- *Implication:* images are not a fixed size, so each pipeline needs a resize step (e.g. resize the shorter side, then crop). A centre crop will cut off the edges of the most elongated images (aspect 0.58 and 1.71).
- *Limitation:* sizes were measured on 300 training images only, not on val or test.